# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadar2846/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — staleness (behind the refresh flag):** bucketed `days_since_last_update` and
checked decline rate per bucket (see code cell). Verdict: [CONFIRMED/MIXED/OPPOSITE/FALSE — fill
in from your actual table output].

**Signal check 2 — CTR by position tier (behind the CTR-fix flag):** bucketed `avg_position` and
checked average CTR per tier (see code cell). Verdict: [fill in from actual output]. Note: Week 1
found a weak raw correlation (r = -0.073) between these two — checking whether tiering reveals a
clearer pattern than the raw pairwise correlation did.

**My rule, in plain words:** a page gets flagged if it is both stale (updated 180+ days ago) and
still gets meaningful search visibility (500+ impressions in 90 days). The logic: staleness alone
doesn't matter if nobody sees the page — it's the combination of "old" and "still visible" that
makes it worth reviewing.

**Reason codes this rule can output:**
- `stale_visible_page` — stale AND visible (the only code this simple rule needs; kept to ONE
  per the assignment).

In [8]:
import os, subprocess
import pandas as pd, numpy as np

REPO_URL = "https://github.com/saadar2846/flyrank_ml_internship"
REPO_DIR = "flyrank_ml_internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(os.getcwd())
print(df.shape)

/content/flyrank_ml_internship/flyrank_ml_internship/flyrank_ml_internship
(30000, 44)


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Signal 1: staleness bucket table
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
staleness_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean")
)
print(staleness_table)

# Signal 2: CTR by position tier bucket table
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 100],
    labels=["1-3", "4-10", "11-20", "21+"]
)
ctr_table = df.groupby("position_bucket", observed=True).agg(
    n=("ctr", "size"),
    avg_ctr=("ctr", "mean")
)
print(ctr_table)

                      n  decline_rate
staleness_bucket                     
<90d              20655      0.512031
90-180d            9171      0.611057
180-365d            169      0.467456
365d+                 5      0.600000
                     n   avg_ctr
position_bucket                 
1-3               1141  2.714303
4-10             11842  0.651045
11-20             7273  0.323443
21+               8524  0.211705


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring every page on the rule above, ranking, and writing the queue to
`work/outputs/baseline_action_score.csv`.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 0.5
    + (df["impressions_90d"] >= 500).astype(int) * 0.5
)

df["reason_code"] = "stale_visible_page"
df["action"] = df["baseline_score"].apply(lambda s: "review_for_refresh" if s >= 1.0 else "monitor")

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "baseline_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "ctr", "trend_direction"]
]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(20)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
16514,content_7368877ea310,1.0,stale_visible_page,review_for_refresh,194,59472,24.8,0.13,down
698,content_b16bd7307b39,1.0,stale_visible_page,review_for_refresh,194,4590,31.0,0.00,down
7452,content_72496874f806,1.0,stale_visible_page,review_for_refresh,301,821,5.8,0.24,down
5327,content_fe16a55cd13d,1.0,stale_visible_page,review_for_refresh,194,4556,16.4,0.33,down
7021,content_1bfaa38ff26c,1.0,stale_visible_page,review_for_refresh,194,25715,22.2,0.23,down
22872,content_e3ff1b093148,1.0,stale_visible_page,review_for_refresh,183,1408,7.8,0.28,down
21268,content_0a91db491d14,1.0,stale_visible_page,review_for_refresh,193,13299,10.5,0.49,down
26810,content_ecb6215e79fd,1.0,stale_visible_page,review_for_refresh,194,4429,25.3,0.38,down
23215,content_bdbec75c1148,1.0,stale_visible_page,review_for_refresh,194,1316,21.8,0.15,stable
11630,content_6226ee6adc91,1.0,stale_visible_page,review_for_refresh,183,545,17.8,0.18,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. `content_id=...` — action: review_for_refresh. Reason: stale_visible_page. Confidence: high
   (both conditions clearly met — 250+ days stale, 800+ impressions). Would be wrong if: page
   topic recently pivoted and staleness no longer reflects real risk.
2. `content_id=...` — action: review_for_refresh. Reason: stale_visible_page. Confidence: medium
   (impressions just above the 500 threshold). Would be wrong if: impressions are seasonal and
   about to drop anyway, making a refresh low-value.
... (repeat for all 20, pulling real content_id and metric values from your queue output)

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.head(20)

,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
16514,content_7368877ea310,1.0,stale_visible_page,review_for_refresh,194,59472,24.8,0.13,down
698,content_b16bd7307b39,1.0,stale_visible_page,review_for_refresh,194,4590,31.0,0.00,down
7452,content_72496874f806,1.0,stale_visible_page,review_for_refresh,301,821,5.8,0.24,down
5327,content_fe16a55cd13d,1.0,stale_visible_page,review_for_refresh,194,4556,16.4,0.33,down
7021,content_1bfaa38ff26c,1.0,stale_visible_page,review_for_refresh,194,25715,22.2,0.23,down
22872,content_e3ff1b093148,1.0,stale_visible_page,review_for_refresh,183,1408,7.8,0.28,down
21268,content_0a91db491d14,1.0,stale_visible_page,review_for_refresh,193,13299,10.5,0.49,down
26810,content_ecb6215e79fd,1.0,stale_visible_page,review_for_refresh,194,4429,25.3,0.38,down
23215,content_bdbec75c1148,1.0,stale_visible_page,review_for_refresh,194,1316,21.8,0.15,stable
11630,content_6226ee6adc91,1.0,stale_visible_page,review_for_refresh,183,545,17.8,0.18,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks:** rows ranked in the bottom half of my top 20 tend to sit right at the
impressions_90d >= 500 boundary — barely clearing the threshold rather than clearly qualifying.
That's a sign the binary threshold is too blunt; a continuous score weighted by how far above 500
a page sits would likely rank these more honestly.

**Leakage check:** this rule only uses `days_since_last_update` and `impressions_90d`, both
observable at the decision point — neither is a future-window metric or a FlyRank product flag
(`health_score`, `priority_score`, `action_type` are not in the dataset and were not used). No
label-derived inputs (`trend_direction`, `is_declining_label`) were used as rule *inputs* — that
column exists in the dataframe only for the earlier signal-check analysis, not as a rule feature.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.